In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

import pandas as pd
import numpy as np
import joblib as jl
import json
import shap
import os

In [3]:
input_dir = "../data/processed" 

In [4]:
# Load data
X_train = pd.read_csv(f"{input_dir}/small_X_train.csv")
y_train = pd.read_csv(f"{input_dir}/small_y_train.csv")

X_val = pd.read_csv(f"{input_dir}/small_X_val.csv")
y_val = pd.read_csv(f"{input_dir}/small_y_val.csv")

X_test = pd.read_csv(f"{input_dir}/small_X_test.csv")
y_test = pd.read_csv(f"{input_dir}/small_y_test.csv")

In [5]:
X_train = X_train.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)
X_val = X_val.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)
X_test = X_test.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)

y_train = y_train.drop(columns = "tf").to_numpy().ravel()
y_val = y_val.drop(columns = "tf").to_numpy().ravel()
y_test = y_test.drop(columns = "tf").to_numpy().ravel()

In [6]:
random_state = 23
max_iter = 2000
model_dir = "../models"

In [13]:
# Define candidate models
models = {
    "LR":
        (LogisticRegression(penalty = "l2",
                           solver = "lbfgs",
                           max_iter = max_iter,
                           random_state = random_state),
        {"classifier__C": [0.05, 0.2, 1.0, 5.0]}),
    "ANN":
        (MLPClassifier(hidden_layer_sizes = (64, 32, 16), # tapering
                      early_stopping = False, # validation on val
                      max_iter = max_iter,
                      random_state = random_state),
        {"classifier__alpha": [1e-4, 1e-3, 1e-2, 1e-1]}),
    "SVM": # LinearSVC has no predict_proba by default
        (CalibratedClassifierCV(estimator = LinearSVC(penalty = "l2",
                                                     dual = False,
                                                     max_iter = max_iter,
                                                     random_state = random_state),
                                cv = 3,
                                ensemble = False),
        {"classifier__estimator__C": [0.05, 0.2, 1.0, 5.0]}),
    "RF":
        (RandomForestClassifier(n_estimators = 200,
                            max_features = "sqrt",
                            n_jobs = 4, # based on machine specs
                            random_state = random_state),
        {"classifier__max_depth": [5, 10, None],
         "classifier__min_samples_leaf": [2, 5, 10],
         "classifier__ccp_alpha": [0.0, 0.005, 0.01]})
}

In [20]:
# for checking if parameter is JSON serializable
def is_json_serializable(x):
    try:
        json.dumps(x)
        return True
    except:
        return False 

In [ ]:
# set up train-val-test pipeline
def train_val_test(name, model, grid):
    best_val_auc = -np.inf
    best_model = None
    best_summary = None
    
    for params in ParameterGrid(grid):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", model)
        ])
    
        pipe.set_params(**params)
        pipe.fit(X_train, y_train)
        val_auc = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1]) # positive only
    
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model = pipe
            best_summary = params
    
    test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{name} | {best_summary}")
    print(f"ROC-AUC: {test_auc}")
    
    y_pred = best_model.predict(X_test)
    print(classification_report(y_test,
                                y_pred,
                                labels = [True, False],
                                target_names = ["TFBS", "Non-TFBS"]
                                ))
    
    print(confusion_matrix(y_test, y_pred))

    # save model and params
    jl.dump(best_model, f"{model_dir}/{name}.pkl", compress = True)
    with open(f"{model_dir}/{name}.json", "w") as params_file:
        params_dict = {k: v for k, v in best_model.get_params(deep = True).items() if is_json_serializable(v)} 
        json.dump(params_dict, params_file, indent = 4)

In [22]:
# Find best Logistic Regression model
name = "LR"
model, grid = models[name]
train_val_test(name, model, grid)

LR | {'classifier__C': 5.0}
ROC-AUC: 0.8094442848600673
              precision    recall  f1-score   support

        TFBS       0.74      0.73      0.74     35912
    Non-TFBS       0.73      0.75      0.74     35796

    accuracy                           0.74     71708
   macro avg       0.74      0.74      0.74     71708
weighted avg       0.74      0.74      0.74     71708

[[26690  9106]
 [ 9692 26220]]


In [23]:
# Find best Artificial Neural Network model
name = "ANN"
model, grid = models[name]
train_val_test(name, model, grid)

ANN | {'classifier__alpha': 0.001}
ROC-AUC: 0.8752214132105396
              precision    recall  f1-score   support

        TFBS       0.80      0.78      0.79     35912
    Non-TFBS       0.79      0.80      0.79     35796

    accuracy                           0.79     71708
   macro avg       0.79      0.79      0.79     71708
weighted avg       0.79      0.79      0.79     71708

[[28677  7119]
 [ 7762 28150]]


In [29]:
# Find best Support Vector Machine model
name = "SVM"
model, grid = models[name]
train_val_test(name, model, grid)

SVM | {'classifier__estimator__C': 5.0}
ROC-AUC: 0.8099210380007638
              precision    recall  f1-score   support

        TFBS       0.74      0.73      0.74     35912
    Non-TFBS       0.73      0.75      0.74     35796

    accuracy                           0.74     71708
   macro avg       0.74      0.74      0.74     71708
weighted avg       0.74      0.74      0.74     71708

[[26731  9065]
 [ 9654 26258]]


In [31]:
# Find best Random Forest model
name = "RF"
model, grid = models[name]
train_val_test(name, model, grid)

RF | {'classifier__ccp_alpha': 0.0, 'classifier__max_depth': None, 'classifier__min_samples_leaf': 2}
ROC-AUC: 0.7176752352368727
              precision    recall  f1-score   support

        TFBS       0.66      0.65      0.66     35912
    Non-TFBS       0.65      0.66      0.66     35796

    accuracy                           0.66     71708
   macro avg       0.66      0.66      0.66     71708
weighted avg       0.66      0.66      0.66     71708

[[23631 12165]
 [12497 23415]]


In [ ]:
# Load the best ANN model and explain top 20 features
best_model = jl.load(f"{model_dir}/ANN.pkl")
background = shap.sample(X_train, 100, random_state = random_state)
X_test_sample = shap.sample(X_test, 1000, random_state = random_state)

explainer = shap.Explainer(best_model.predict_proba, background)
shap_values = explainer(X_test_sample)
importance = np.abs(shap_values.values[:, :, 1]).mean(axis = 0)

feature_ranking = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importance
}).sort_values(by = "importance", ascending = False)

print(feature_ranking.head(20))
feature_ranking.to_csv(f"{model_dir}/ANN.feature_importance.csv", index = False)

PermutationExplainer explainer: 1001it [03:13,  4.88it/s]                          


                               feature  importance
55                        cgr_skewness    0.095006
141           zcurve_none_levated_peak    0.057339
8             atomic_none_levated_peak    0.053965
43                         cgr_maximum    0.051119
38                       cgr_amplitude    0.050527
42                        cgr_kurtosis    0.044466
40        cgr_coefficient_of_variation    0.039655
53       cgr_sample_standard_deviation    0.037077
52   cgr_population_standard_deviation    0.036938
39                         cgr_average    0.033635
143                zcurve_percentile15    0.031749
149    zcurve_semi_interquartile_range    0.030246
54        cgr_semi_interquartile_range    0.029723
136         zcurve_interquartile_range    0.029698
41             cgr_interquartile_range    0.029624
47                            cgr_peak    0.028486
142                        zcurve_peak    0.028214
51                    cgr_percentile75    0.027176
135    zcurve_coefficient_of_va